In [1]:
import requests
import pandas as pd
import numpy as np
from pathlib import Path

In [15]:
START_YEAR = 1992
END_YEAR = 2022

COUNTRIES = {
    "HUN": "Hungary",
    "CZE": "Czechia",
    "POL": "Poland",
    "ROU": "Romania",
    "BGR": "Bulgaria",
    "GRC": "Greece",
}

INDICATORS = {
    "NY.GDP.MKTP.KD": "gdp",
    "SL.EMP.TOTL.SP.FE.ZS": "erf",
    "SP.URB.TOTL.IN.ZS": "up",
    "NE.TRD.GNFS.ZS": "trd",
    "NE.CON.TOTL.KD": "fce",
}


def fetch_indicator(indicator_code):
    """
    Download one World Bank indicator for all selected countries and years.
    """

    country_string = ";".join(COUNTRIES.keys())

    url = (
        f"https://api.worldbank.org/v2/country/{country_string}"
        f"/indicator/{indicator_code}"
    )

    params = {
        "source": 2,
        "date": f"{START_YEAR}:{END_YEAR}",
        "format": "json",
        "per_page": 20000,
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()

    data = response.json()

    if len(data) < 2 or data[1] is None:
        raise ValueError(f"No data returned for indicator {indicator_code}")

    rows = []

    for item in data[1]:
        rows.append({
            "country_code": item["countryiso3code"],
            "country": COUNTRIES.get(item["countryiso3code"], item["country"]["value"]),
            "year": int(item["date"]),
            "indicator_code": indicator_code,
            "indicator_name": item["indicator"]["value"],
            "value": item["value"],
        })

    return pd.DataFrame(rows)


def collect_data():
    """
    Download all indicators and create long and wide datasets.
    """

    all_data = []
    total_indicators = len(INDICATORS)

    for i, (indicator_code, variable_name) in enumerate(INDICATORS.items(), start=1):
        print(f"[{i}/{total_indicators}] Downloading {indicator_code}: {variable_name}")

        df = fetch_indicator(indicator_code)
        df["variable"] = variable_name

        print(f"[{i}/{total_indicators}] Finished {variable_name}: {len(df)} rows collected")

        all_data.append(df)

    print("\nCombining downloaded indicators...")

    long_data = pd.concat(all_data, ignore_index=True)

    wide_data = (
        long_data
        .pivot_table(
            index=["country_code", "country", "year"],
            columns="variable",
            values="value",
            aggfunc="first"
        )
        .reset_index()
    )

    wide_data = wide_data.sort_values(["country_code", "year"]).reset_index(drop=True)

    print("Data collection finished.")

    return long_data, wide_data


long_data, wide_data = collect_data()

print("\nPreview of wide dataset:")
print(wide_data.head())

print("\nDataset shape:")
print(wide_data.shape)

[1/5] Downloading NY.GDP.MKTP.KD: gdp
[1/5] Finished gdp: 186 rows collected
[2/5] Downloading SL.EMP.TOTL.SP.FE.ZS: erf
[2/5] Finished erf: 186 rows collected
[3/5] Downloading SP.URB.TOTL.IN.ZS: up
[3/5] Finished up: 186 rows collected
[4/5] Downloading NE.TRD.GNFS.ZS: trd
[4/5] Finished trd: 186 rows collected
[5/5] Downloading NE.CON.TOTL.KD: fce
[5/5] Finished fce: 186 rows collected

Combining downloaded indicators...
Data collection finished.

Preview of wide dataset:
variable country_code   country  year     erf           fce           gdp  \
0                 BGR  Bulgaria  1992  37.889  2.749091e+10  3.275541e+10   
1                 BGR  Bulgaria  1993  37.272  2.651163e+10  3.227056e+10   
2                 BGR  Bulgaria  1994  38.752  2.532331e+10  3.285724e+10   
3                 BGR  Bulgaria  1995  39.384  3.015775e+10  3.380916e+10   
4                 BGR  Bulgaria  1996  38.614  2.948093e+10  3.556906e+10   

variable         trd         up  
0         100.049801  6

In [17]:
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

# Save raw wide data collected from World Bank
wide_data.to_csv(output_dir / "wdi_wide_raw.csv", index=False)


df_wide = wide_data.copy()

model_vars = ["gdp", "erf", "up", "trd", "fce"]

for col in model_vars:
    df_wide[col] = pd.to_numeric(df_wide[col], errors="coerce")

df_wide = df_wide.dropna(subset=model_vars).copy()

for col in model_vars:
    if (df_wide[col] <= 0).any():
        raise ValueError(f"{col} contains zero or negative values, log is not possible.")

df_wide["lgdp"] = np.log(df_wide["gdp"])
df_wide["lerf"] = np.log(df_wide["erf"])
df_wide["lup"] = np.log(df_wide["up"])
df_wide["ltrd"] = np.log(df_wide["trd"])
df_wide["lfce"] = np.log(df_wide["fce"])

df_model = df_wide[
    [
        "country_code",
        "country",
        "year",
        "lgdp",
        "lerf",
        "lup",
        "ltrd",
        "lfce",
    ]
].sort_values(["country_code", "year"]).reset_index(drop=True)

# Save final logged dataset for modelling
df_model.to_csv(output_dir / "wdi_wide_logged.csv", index=False)

print("Saved raw wide dataset to:", output_dir / "wdi_wide_raw.csv")
print("Saved logged modelling dataset to:", output_dir / "wdi_wide_logged.csv")

Saved raw wide dataset to: data\wdi_wide_raw.csv
Saved logged modelling dataset to: data\wdi_wide_logged.csv
